In [ ]:
import autograd.numpy as anp
import numpy as np
import os
os.chdir('../..')
os.getcwd()

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
from benchmarks_august.targets.bnn import bnn_classification
df = pd.read_csv("benchmarks_august/datasets/diabetes.csv")
X = df.drop("Outcome", axis=1).values.astype(float)
y = df["Outcome"].values.astype(float)

# Handle implicit missingness: zero means missing in these columns
for col_idx in [1, 2, 3, 4, 5]:  # Glucose, BP, Skin, Insulin, BMI
    mask = X[:, col_idx] == 0
    X[mask, col_idx] = np.nan
X = np.where(np.isnan(X), np.nanmean(X, axis=0), X)  # mean impute

# Standardize
X = (X - X.mean(axis=0)) / X.std(axis=0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from benchmarks_august.targets.bnn import bnn_classification
from benchmarks_august.samplers.warmstart.bnn import adam_fisher
from benchmarks_august.samplers import build_sampler, apply_preprocess

H=10
layers = [X.shape[1], H, 1]

# 1. Build target (E and gradE come from here)
target = bnn_classification(X_train, y_train, layer_sizes=layers,
    prior={"kind": "layered_gaussian",
           "sigma_w_layers": [1.0, 1.0],
           "sigma_b_layers": [1.0, 1.0]})

# 2. Warmstart (sets x_ref and Sigma_inv)
ws = adam_fisher(target, n_epochs=3000, lr=3e-3, l2_weight=0.001, l1_weight=0.02)
target.x_ref = ws["x_ref"]
target.Sigma_inv = ws["Sigma_inv"]

# 3. Build kappa from target.meta
w_prior = 0.5
kappa = np.empty(target.D)
kappa[~target.meta["weight_mask"]] = 1e6
for l, (w_sl, b_sl) in enumerate(target.meta["slices"]):
    sigma_l = target.meta["sigma_w_layers"][l]
    kappa[w_sl] = (1 - w_prior) / w_prior / (sigma_l * np.sqrt(2 * np.pi))

In [ ]:
np.sum((np.abs(target.x_ref))<0.05)

In [ ]:
sampler = build_sampler("boomerang_pli", target, N=10000,
                        refresh_rate=1.0)
apply_preprocess(sampler, target, {"method": "manual"})
sampler.sample_auto()

In [ ]:
# 4. Build sampler using target.E and target.gradE
sampler_sticky = build_sampler("sticky_boomerang_pli", target, N=10000,
                        kappa=kappa, refresh_rate=1.0, cold_start_threshold=0.05)
apply_preprocess(sampler_sticky, target, {"method": "manual"})
sampler_sticky.sample_auto()

In [ ]:
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path, resample_sticky_pdmp_path
t, x = resample_pdmp_path(sampler, n_samples=50000)
t_sticky, x_sticky = resample_sticky_pdmp_path(sampler_sticky, n_samples=50000)

In [ ]:
from benchmarks_august.analysis.metrics import bnn_performance

bnn_performance(X_train, y_train, X_test, y_test,
    {
        "Adam MAP": target.x_ref.reshape(1, -1),
        "Boomerang PLI": x,
        "Sticky PLI": x_sticky,
    },
    shapes=target.meta["shapes"],
    slices=target.meta["slices"],
    weight_mask=target.meta["weight_mask"])